In [6]:

import numpy as np

np.set_printoptions(suppress=True, linewidth=140)

DATA_PATH = "../data/penguins.csv"


In [7]:

# Exercise 1

P = np.genfromtxt(
    DATA_PATH,
    delimiter=",",
    skip_header=1,
    usecols=(2, 3, 4, 5),
    dtype=float,
    missing_values="NA",
    filling_values=np.nan,
    encoding="utf-8"
)

penguin_species = np.genfromtxt(
    DATA_PATH,
    delimiter=",",
    skip_header=1,
    usecols=0,
    dtype=str,
    encoding="utf-8"
)

print("P shape:", P.shape)
print("penguin_species shape:", penguin_species.shape)


P shape: (344, 4)
penguin_species shape: (344,)


In [8]:

# Exercise 2

print("Total missing values:", np.isnan(P).sum())

print("Missing values per column:",
      np.isnan(P).sum(axis=0))

rows_with_missing = np.isnan(P).any(axis=1)

print("Rows with at least one missing value:",
      rows_with_missing.sum())

print("Rows containing missing values:")
print(P[rows_with_missing])

print("P.mean(axis=0):", P.mean(axis=0))
print("Correct column means:", np.nanmean(P, axis=0))


Total missing values: 8
Missing values per column: [2 2 2 2]
Rows with at least one missing value: 2
Rows containing missing values:
[[nan nan nan nan]
 [nan nan nan nan]]
P.mean(axis=0): [nan nan nan nan]
Correct column means: [  43.92192982   17.15116959  200.91520468 4201.75438596]


The ordinary mean() function does not ignore NaNs. If any value in a column is NaN, that column's mean becomes NaN. Use np.nanmean() to compute means from the observed values.

What we notice: Some penguins have missing measurements. The incomplete rows may have more than one missing measurement, so count rows separately from individual missing cells.

In particular, the assignment notes that two rows have all four measurements missing. For those rows, imputing all four measurements would create artificial records without any observed numerical evidence.

In [9]:

# Exercise 3.1: Mean imputation

mean_values = np.nanmean(P, axis=0)
P_mean = P.copy()

for j in range(P.shape[1]):
    missing = np.isnan(P_mean[:, j])
    P_mean[missing, j] = mean_values[j]


# Exercise 3.2: Median imputation

median_values = np.nanmedian(P, axis=0)
P_median = P.copy()

for j in range(P.shape[1]):
    missing = np.isnan(P_median[:, j])
    P_median[missing, j] = median_values[j]


# Exercise 3.3: Verify no NaNs remain

print("NaNs in P_mean:", np.isnan(P_mean).sum())
print("NaNs in P_median:", np.isnan(P_median).sum())


# Exercise 3.4: Print fill values

feature_names = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

print("Mean fill values:", np.round(mean_values, 4))
print("Median fill values:", np.round(median_values, 4))
print("Absolute differences:", np.round(
    np.abs(mean_values - median_values), 4
))

largest_difference = np.argmax(np.abs(mean_values - median_values))

print("Column with largest difference:",
      feature_names[largest_difference])


NaNs in P_mean: 0
NaNs in P_median: 0
Mean fill values: [  43.9219   17.1512  200.9152 4201.7544]
Median fill values: [  44.45   17.3   197.   4050.  ]
Absolute differences: [  0.5281   0.1488   3.9152 151.7544]
Column with largest difference: body_mass_g


3.5 Should you impute the two completely missing rows?
Answer:

I would generally prefer to drop the two rows for numerical analysis because every measurement is missing. Imputing them would assign four artificial measurements to each penguin, potentially distorting the dataset. If the rows contain useful information in other columns, I would retain that information separately rather than inventing numerical observations.

This is a methodological choice; the assignment asks you to explain your reasoning rather than to remove those rows from P_mean or P_median.

In [10]:

# Exercise 4.1

print("Minimum:", np.min(P_median, axis=0))
print("Maximum:", np.max(P_median, axis=0))


Minimum: [  32.1   13.1  172.  2700. ]
Maximum: [  59.6   21.5  231.  6300. ]


Body mass has values measured in grams, whereas bill measurements are in millimetres. Raw numerical scales differ, so a distance-based method may be dominated by features with larger numerical ranges. Min-max scaling or standardisation can make the feature scales more comparable.

In [13]:

# Exercise 4.2

P_minmax = (
    (P_median - np.min(P_median, axis=0))
    / (np.max(P_median, axis=0) - np.min(P_median, axis=0))
)

print("Minimum after min-max scaling:",
      np.min(P_minmax, axis=0))
print("Maximum after min-max scaling:",
      np.max(P_minmax, axis=0))

print("All minima are zero:",
      np.allclose(np.min(P_minmax, axis=0), 0))

print("All maxima are one:",
      np.allclose(np.max(P_minmax, axis=0), 1))


Minimum after min-max scaling: [0. 0. 0. 0.]
Maximum after min-max scaling: [1. 1. 1. 1.]
All minima are zero: True
All maxima are one: True


In [14]:

# Exercise 4.3

P_z = (
    (P_median - np.mean(P_median, axis=0))
    / np.std(P_median, axis=0)
)

print("Means after z-score:", np.mean(P_z, axis=0))
print("Standard deviations after z-score:",
      np.std(P_z, axis=0))

print("Means approximately zero:",
      np.allclose(np.mean(P_z, axis=0), 0))

print("Standard deviations approximately one:",
      np.allclose(np.std(P_z, axis=0), 1))


Means after z-score: [-0. -0.  0.  0.]
Standard deviations after z-score: [1. 1. 1. 1.]
Means approximately zero: True
Standard deviations approximately one: True


In [15]:

# Exercise 5

values = np.array([1.0, 2.0, 3.0, 4.0, 100.0])

minmax_values = (
    (values - values.min())
    / (values.max() - values.min())
)

zscore_values = (
    (values - values.mean())
    / values.std()
)

print("Min-max scaling:", np.round(minmax_values, 3))
print("Z-score standardisation:", np.round(zscore_values, 3))


Min-max scaling: [0.   0.01 0.02 0.03 1.  ]
Z-score standardisation: [-0.538 -0.513 -0.487 -0.461  1.999]


5.2 What happens to the four normal values?
Under min-max scaling, the values 1, 2, 3, and 4 are compressed into the narrow range 0 to 0.03 because the maximum value is 100.

Under z-score standardisation, those same values become close to each other and slightly below zero. The extreme outlier increases the mean and standard deviation, which affects the standardised values.

5.3 Which method should you choose with outliers?
Answer:

Neither min-max scaling nor z-score standardisation is robust to extreme outliers. Min-max scaling compresses most normal values into a narrow range, while z-score standardisation is affected by the outlier's influence on the mean and standard deviation. If outliers are present, I would consider robust scaling based on the median and interquartile range, after checking whether the outliers are genuine observations or data errors.

In [16]:

# Exercise 6.1

island, sex = np.genfromtxt(
    DATA_PATH,
    delimiter=",",
    skip_header=1,
    usecols=(1, 6),
    dtype=str,
    missing_values="NA",
    filling_values="",
    encoding="utf-8",
    autostrip=True,
    unpack=True
)

print("Island shape:", island.shape)
print("Sex shape:", sex.shape)


Island shape: (344,)
Sex shape: (344,)


In [17]:

# Exercise 6.2

species_categories, species_codes = np.unique(
    penguin_species,
    return_inverse=True
)

island_categories, island_codes = np.unique(
    island,
    return_inverse=True
)

print("Species categories:", species_categories)
print("First 5 species codes:", species_codes[:5])

print("Island categories:", island_categories)
print("First 5 island codes:", island_codes[:5])


Species categories: ['Adelie' 'Chinstrap' 'Gentoo']
First 5 species codes: [0 0 0 0 0]
Island categories: ['Biscoe' 'Dream' 'Torgersen']
First 5 island codes: [2 2 2 2 2]


In [18]:

# Exercise 6.3

species_onehot = np.eye(
    len(species_categories),
    dtype=int
)[species_codes]

print("Species one-hot shape:", species_onehot.shape)
print("First 5 rows:")
print(species_onehot[:5])

print("Row sums:", species_onehot.sum(axis=1))
print("Column sums:", species_onehot.sum(axis=0))

print("Every row sums to one:",
      np.all(species_onehot.sum(axis=1) == 1))


Species one-hot shape: (344, 3)
First 5 rows:
[[1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]]
Row sums: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
Column sums: [152  68 124]
Every row sums to one: True


Interpretation:
- Each row sums to 1 because each penguin belongs to exactly one species.
- The three column sums give the number of penguins belonging to each species, in the order shown in species_categories.

In [19]:

# Exercise 6.4

sex_categories, sex_counts = np.unique(
    sex,
    return_counts=True
)

print("Sex categories and counts:")
print(dict(zip(sex_categories.tolist(), sex_counts.tolist())))

print("Missing sex values:", np.sum(sex == ""))


Sex categories and counts:
{'NA': 11, 'female': 165, 'male': 168}
Missing sex values: 0


Missing sex values should not be treated as a genuine sex category. I would either retain them as missing until the analysis requires a decision, or impute them using the most frequent observed category if appropriate for the task. I would also consider dropping rows if the missingness makes them unsuitable for the intended analysis.

In [20]:

# Exercise 6.5

P_combined = np.hstack([
    P_z,
    species_onehot
])

print("Combined feature matrix shape:", P_combined.shape)


Combined feature matrix shape: (344, 7)
